# 01 — Cleaning the Online Retail II Dataset

**Project:** Cohort & Retention Analysis  
**Business question:** *Which customers stopped buying, when did they leave, and why?*

This notebook produces a clean transaction-level dataset that the next notebooks (cohort matrix, churn drivers) will build on.

## Cleaning rules (Phase 2)
1. Remove rows with no `CustomerID` (can't track cohorts without it)
2. Remove cancelled orders (`Invoice` starting with `C`)
3. Remove rows where `Quantity` or `Price` is **≤ 0**
4. Add a `Revenue` column = `Quantity × Price`
5. Parse `InvoiceDate` as a proper datetime

**Input:** `data/raw/online_retail_II.xlsx` (two sheets: 2009–2010 and 2010–2011)  
**Output:** `data/clean/online_retail_clean.parquet` (+ a small `.csv` sample)

## 0. Imports & paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH     = PROJECT_ROOT / "data" / "raw"   / "online_retail_II.xlsx"
CLEAN_DIR    = PROJECT_ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw file exists:", RAW_PATH.exists(), "->", RAW_PATH)

Project root: C:\Users\Pitu\Desktop\Claude\Cohort & Retention Analysis
Raw file exists: True -> C:\Users\Pitu\Desktop\Claude\Cohort & Retention Analysis\data\raw\online_retail_II.xlsx


## 1. Load the raw data

Online Retail II ships as **one Excel file with two sheets** (one per year). We read both and concatenate them into a single DataFrame.

> Reading ~1M rows from `.xlsx` is slow (30–90 seconds). That's fine — we only do it once and cache the result as Parquet at the end.

In [2]:
sheets = pd.read_excel(RAW_PATH, sheet_name=None, engine="openpyxl")
print("Sheets found:", list(sheets.keys()))
for name, frame in sheets.items():
    print(f"  {name:<20} rows={len(frame):>8,}  cols={list(frame.columns)}")

df_raw = pd.concat(sheets.values(), ignore_index=True)
print("\nCombined raw shape:", df_raw.shape)
df_raw.head()

Sheets found: ['Year 2009-2010', 'Year 2010-2011']
  Year 2009-2010       rows= 525,461  cols=['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
  Year 2010-2011       rows= 541,910  cols=['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Combined raw shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Standardize column names

The Online Retail II columns are `Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID`, `Country`.

We rename `Customer ID → CustomerID` so the rest of the project can use a clean identifier.

In [3]:
df_raw = df_raw.rename(columns={"Customer ID": "CustomerID"})
df_raw.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
CustomerID            float64
Country                   str
dtype: object

## 2. Diagnose what's dirty

Before cleaning, let's quantify each issue. This is the "why this step matters" view that you'll reuse in the Tableau story.

In [4]:
n_total      = len(df_raw)
n_no_cust    = df_raw["CustomerID"].isna().sum()
n_cancelled  = df_raw["Invoice"].astype(str).str.startswith("C").sum()
n_qty_bad    = (df_raw["Quantity"] <= 0).sum()
n_price_bad  = (df_raw["Price"]    <= 0).sum()

diag = pd.DataFrame({
    "issue":  ["total rows", "missing CustomerID", "cancelled (Invoice starts with C)", "Quantity <= 0", "Price <= 0"],
    "rows":   [n_total, n_no_cust, n_cancelled, n_qty_bad, n_price_bad],
    "pct_of_total": [1.0, n_no_cust/n_total, n_cancelled/n_total, n_qty_bad/n_total, n_price_bad/n_total],
})
diag["pct_of_total"] = (diag["pct_of_total"] * 100).round(2).astype(str) + " %"
diag

,issue,rows,pct_of_total
0,total rows,1067371,100.0 %
1,missing CustomerID,243007,22.77 %
2,cancelled (Invoice starts with C),19494,1.83 %
3,Quantity <= 0,22950,2.15 %
4,Price <= 0,6207,0.58 %


## 3. Apply cleaning rules

We apply the filters one by one and log how many rows survive each step. Doing it sequentially (instead of one big boolean mask) makes the funnel visible.

In [5]:
df = df_raw.copy()
funnel = [("raw", len(df))]

# Rule 1: drop rows with no CustomerID
df = df[df["CustomerID"].notna()].copy()
funnel.append(("after drop missing CustomerID", len(df)))

# Rule 2: drop cancelled orders (Invoice starting with 'C')
df["Invoice"] = df["Invoice"].astype(str)
df = df[~df["Invoice"].str.startswith("C")].copy()
funnel.append(("after drop cancelled invoices", len(df)))

# Rule 3: drop non-positive Quantity or Price
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)].copy()
funnel.append(("after drop non-positive Qty/Price", len(df)))

pd.DataFrame(funnel, columns=["step", "rows_remaining"]).assign(
    pct_of_raw=lambda d: (d["rows_remaining"] / d["rows_remaining"].iloc[0] * 100).round(2)
)

,step,rows_remaining,pct_of_raw
0,raw,1067371,100.00
1,after drop missing CustomerID,824364,77.23
2,after drop cancelled invoices,805620,75.48
3,after drop non-positive Qty/Price,805549,75.47


## 4. Add `Revenue` and enforce types

- `Revenue = Quantity * Price`
- `InvoiceDate` → real `datetime64[ns]` (Excel sometimes loads it as a string)
- `CustomerID` → integer (Excel reads it as float because of the original NaNs)

In [6]:
df["Revenue"]     = df["Quantity"] * df["Price"]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"]  = df["CustomerID"].astype("int64")

# Force text columns to string — some StockCodes mix digits and letters
# (e.g. '79323P'), so without this Arrow tries to infer int64 and the
# Parquet save fails.
for col in ["Invoice", "StockCode", "Description", "Country"]:
    df[col] = df[col].astype("string")

# Drop the (rare) rows whose date didn't parse
bad_dates = df["InvoiceDate"].isna().sum()
if bad_dates:
    print(f"Dropping {bad_dates} rows with unparseable InvoiceDate")
    df = df[df["InvoiceDate"].notna()].copy()

df.dtypes

Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
CustomerID              int64
Country                string
Revenue               float64
dtype: object

## 5. Sanity checks

Quick confirmations before saving:
- No nulls in the columns we care about
- `Quantity`, `Price`, `Revenue` are all strictly positive
- Date range covers ~2 years
- Customer count looks plausible (~5–6k unique customers is normal for this dataset)

In [7]:
checks = {
    "rows":                     len(df),
    "unique invoices":          df["Invoice"].nunique(),
    "unique customers":         df["CustomerID"].nunique(),
    "unique products":          df["StockCode"].nunique(),
    "date min":                 df["InvoiceDate"].min(),
    "date max":                 df["InvoiceDate"].max(),
    "nulls in key cols":        int(df[["Invoice","CustomerID","InvoiceDate","Quantity","Price","Revenue"]].isna().sum().sum()),
    "min Quantity":             int(df["Quantity"].min()),
    "min Price":                float(df["Price"].min()),
    "min Revenue":              float(df["Revenue"].min()),
    "total Revenue (£)":        round(df["Revenue"].sum(), 2),
}
pd.Series(checks, name="value").to_frame()

,value
rows,805549
unique invoices,36969
unique customers,5878
unique products,4631
date min,2009-12-01 07:45:00
date max,2011-12-09 12:50:00
nulls in key cols,0
min Quantity,1
min Price,0.001
min Revenue,0.001


In [8]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


## 6. Save the clean dataset

We save in **Parquet** (fast + typed) for the next notebooks, and a small CSV sample for quick eyeballing or sharing.

In [9]:
parquet_path = CLEAN_DIR / "online_retail_clean.parquet"
sample_path  = CLEAN_DIR / "online_retail_clean_sample.csv"

df.to_parquet(parquet_path, index=False)
df.sample(min(5000, len(df)), random_state=42).to_csv(sample_path, index=False)

print("Saved:")
print(" ", parquet_path, f"({parquet_path.stat().st_size/1e6:.1f} MB)")
print(" ", sample_path,  f"({sample_path.stat().st_size/1e6:.1f} MB)")

Saved:
  C:\Users\Pitu\Desktop\Claude\Cohort & Retention Analysis\data\clean\online_retail_clean.parquet (6.4 MB)
  C:\Users\Pitu\Desktop\Claude\Cohort & Retention Analysis\data\clean\online_retail_clean_sample.csv (0.5 MB)


## Next: Phase 3 — Cohort analysis

In `02_cohort.ipynb` we'll:
1. Define each customer's **cohort** = month of their first purchase.
2. Compute **months since first purchase** for every transaction.
3. Build a cohort retention matrix (rows = cohort month, cols = months since acquisition, values = % of cohort still active).
4. Export the matrix as a CSV for Tableau.